In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType,DateType

races_schema = StructType(fields=
                             [
                             StructField("raceId", IntegerType(), False),
                             StructField("year", IntegerType(), True),
                             StructField("round", IntegerType(), True),
                             StructField("circuitId", IntegerType(), True),
                             StructField("name", StringType(), True),
                             StructField("date", DateType(), True),
                             StructField("time", StringType(), True),
                             StructField("url", StringType(), True)
                             ])


In [0]:
# df = spark.read\
#     .schema(races_schema)\
#     .option("header", True)\
#     .csv("abfss://demofiles@formula1adls.dfs.core.windows.net/source_files/races.csv")
# df.count()

In [0]:
# abfss://demofiles@formula1adls.dfs.core.windows.net=>
# /Volumes/formula1_dev/bronze/demo_volume

In [0]:
df_volume = spark.read\
    .schema(races_schema)\
    .option("header", True)\
    .csv("/Volumes/formula1_dev/bronze/demo_volume/source_files/races.csv")
    # .csv("abfss://demofiles@formula1adls.dfs.core.windows.net/source_files/races.csv")
df_volume.display()

In [0]:
#add new column to the data frame 
from pyspark.sql.functions import current_timestamp,current_date
df_sample= df_volume.withColumnRenamed("raceId", "race_id")\
    .withColumnRenamed("circuitId", "circuit_id")\
        .withColumn("ingestion_timestamp", current_timestamp())\
            .withColumn("ingestion_date", current_date())\
                .drop("url")

In [0]:
df_sample.display()

In [0]:
df_sample.write.mode("overwrite").option("mergeSchema", True).format("delta").option("path", "abfss://raw@formula1adls.dfs.core.windows.net/races_sample").saveAsTable("formula1_dev.bronze.race")

In [0]:
%sql
select * from formula1_dev.bronze.race